# Merge & Report (Notebook 4 / 4)

Run this notebook **after** notebooks 01, 02, and 03 have all finished.

It:
1. Copies the three method cache files from Drive into the local cache
2. Runs the full experiment (all methods) — every result is a cache hit, no timing
3. Generates the interactive HTML report
4. Saves the HTML to Drive and offers a download link

### Expected Drive folder contents before running
```
MyDrive/treebranchmarks/woodelfhd_sweep/
  woodelf_hd.json        ← written by notebook 01
  original_woodelf.json  ← written by notebook 02
  shap.json              ← written by notebook 03
```

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Step 2: Configure paths ──────────────────────────────────────────────────
import pathlib

DRIVE_FOLDER = pathlib.Path('/content/drive/MyDrive/treebranchmarks/woodelfhd_sweep')
REPORT_HTML  = DRIVE_FOLDER / 'woodelfhd_depth_sweep_experiment.html'

DRIVE_CACHES = {
    'woodelf_hd':       DRIVE_FOLDER / 'woodelf_hd.json',
    'original_woodelf': DRIVE_FOLDER / 'original_woodelf.json',
    'shap':             DRIVE_FOLDER / 'shap.json',
}

missing = [str(p) for p in DRIVE_CACHES.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        'The following method cache files are missing — run the corresponding '
        'notebooks first:\n' + '\n'.join(missing)
    )
print('All three method cache files found.')

In [ ]:
# ── Step 3: Clone treebranchmarks and install ────────────────────────────────
# woodelf_explainer is needed by treebranchmarks even though this notebook
# only calls HtmlGenerator (which does not import woodelf directly).

TREEBRANCHMARKS_URL = 'YOUR_TREEBRANCHMARKS_REPO_URL'
WOODELF_URL         = 'YOUR_WOODELF_REPO_URL'

!git clone {TREEBRANCHMARKS_URL} /content/treebranchmarks
!git clone {WOODELF_URL}         /content/woodelf_explainer

!pip install -q -e /content/woodelf_explainer
!pip install -q -e /content/treebranchmarks

In [ ]:
# ── Step 4: Copy method cache files into local cache ─────────────────────────
import shutil, pathlib

cache_dir = pathlib.Path('/content/treebranchmarks/cache/method_results/woodelfhd_depth_sweep_experiment')
cache_dir.mkdir(parents=True, exist_ok=True)

for method_name, drive_path in DRIVE_CACHES.items():
    dest = cache_dir / f'{method_name}.json'
    shutil.copy(drive_path, dest)
    print(f'Copied {method_name}.json ({drive_path.stat().st_size // 1024} KB)')

In [ ]:
# ── Step 5: Run experiment (all cache hits) + generate HTML ──────────────────
import os, shutil

os.chdir('/content/treebranchmarks')

from benchmarks.woodelfhd_depth_sweep_experiment import build_experiment

exp = build_experiment()
exp.run()
html_path = exp.generate_html()

shutil.copy(html_path, REPORT_HTML)
print(f'HTML report saved to Drive: {REPORT_HTML}')

In [ ]:
# ── Step 6: Download the HTML report ─────────────────────────────────────────
from google.colab import files
files.download(str(REPORT_HTML))